# Approach 2: chunked Semantic IRT — item and benchmark holdouts

This follows the Approach 1 sequence: inspect data → cache frozen embeddings → edit training settings → split by item/prompt → train → compare baselines → plot curves.

We reuse **Steven's exact saved 1,000 responses**, not Approach 1's larger dataset. The model remains his 4PL Semantic IRT with 64-unit hidden layers and row-weighted Brier/MSE training. Chunking retains at most 2,048 content tokens in 512-token windows and pools to one 384-dimensional vector.

**Item holdout:** reserve 20% of item/prompt groups for validation. Repeated attempts, repeated item IDs and identical nonempty prompt text stay together.

**Benchmark holdout:** keep one entire benchmark out of training and checkpoint selection. Within the remaining benchmarks, reserve 20% of item/prompt groups for internal validation; score the held-out benchmark once after restoring the selected checkpoint. Shared prompts crossing the benchmark boundary are purged from the training pool.

All responses are scored. An unseen subject name uses Steven's existing population-average ability fallback. We report its count rather than silently dropping it. Validation selects the checkpoint; it is **not an untouched final test**. No submission is uploaded or all-data production model trained here.


In [ ]:
# Setup: run from this repository in the existing project Python environment.
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
from pathlib import Path
import hashlib
import importlib.metadata
import json
import platform
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from holdout_training import (
    load_samples, make_split, run_fold, score, learning_curves, comparison_plot,
)
from models.Semantic_IRT import Semantic_IRT
from models.text_chunking import TEXT_ENCODING

WORKSPACE = Path.cwd()
assert (WORKSPACE / "samples.json").is_file(), "Launch this notebook from the PSN-IRT directory."
torch.set_num_threads(2)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Python:", sys.executable)
print("Training device:", DEVICE)
print("Torch:", torch.__version__)


## Fixed prompt embeddings

Use the existing local MiniLM encoder when available, otherwise the model already named by Steven's code. `samples_chunked.json` is reused only when response identities, prompts, labels and encoding settings match. Original `samples.json` stays unchanged. Frozen text embeddings use no response labels.


In [ ]:
# Encode once, then reuse this disk cache for later training runs.
LOCAL_ENCODER = WORKSPACE.parent / "approach1-starter/resources/text_encoder"
ENCODER_SOURCE = str(LOCAL_ENCODER) if LOCAL_ENCODER.is_dir() else TEXT_ENCODING["model"]
frame, item_vectors, source_hash = load_samples(
    WORKSPACE / "samples.json", WORKSPACE / "samples_chunked.json", ENCODER_SOURCE, DEVICE
)
dataset_overview = frame.groupby("benchmark_id").agg(
    responses=("response", "size"), items=("item_id", "nunique"),
    subjects=("normalized_name", "nunique"), observed_success_rate=("response", "mean"),
)
display(dataset_overview)
print("Total responses:", len(frame), "| Distinct items:", frame.item_id.nunique(),
      "| Item/prompt groups:", frame.item_group.nunique())
print("Encoding:", TEXT_ENCODING)


## Training settings and architecture

Keep the original architecture, AdamW learning rate/weight decay and maximum 100 epochs. Use 512-response batches. As in Approach 1, restore the best **equal subject-benchmark-pair validation Brier** checkpoint; stop after 10 epochs without improvement. The learning-rate scheduler also sees internal validation only.

Choose a new `RUN_NAME` for each experiment. Existing runs are never overwritten. To revisit graphs, run Setup, this settings cell and the saved-results/plot cells; skip embeddings and training.


In [ ]:
# Edit these settings for a subsequent run; use a fresh RUN_NAME.
RUN_NAME = "chunked_holdouts_20261107_v1"
RUN_DIR = WORKSPACE / "results/chunked/experiments" / RUN_NAME
SETTINGS = {
    "seed": 5814,
    "epochs": 100,
    "batch_size": 512,
    "learning_rate": 3e-4,
    "weight_decay": 1e-5,
    "patience": 10,
    "validation_fraction": 0.20,
}
print(Semantic_IRT(num_students=46, item_embed_dim=384, hidden_dim=64, model_type="4pl"))
print("Run directory:", RUN_DIR)


## Fit an item holdout and all four benchmark holdouts

The helper checks that item/prompt groups do not cross training, internal validation or outer benchmark boundaries. Every response's partition is saved in `split.csv`. Training and validation curves use dropout-free epoch-end weights and the same metric definitions. Brier is squared probability error: **lower is better**. Equal-pair Brier averages within each subject-ID/benchmark pair and then gives every pair equal weight, matching Approach 1's reporting.

The benchmark result is not monitored during training. Never compare these scores directly with Steven's old fold scores: his original shuffled concatenation mixed held-out benchmark rows into training; that split is corrected on this branch.


In [ ]:
# Train. This creates a new run; use the saved CSVs below to replot without retraining.
RUN_DIR.mkdir(parents=True, exist_ok=False)
manifest = {
    "dataset_sha256": source_hash,
    "chunked_samples_sha256": hashlib.sha256((WORKSPACE / "samples_chunked.json").read_bytes()).hexdigest(),
    "source_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "source_sha256": {name: hashlib.sha256((WORKSPACE / name).read_bytes()).hexdigest()
                      for name in ("holdout_training.py", "models/Semantic_IRT.py", "models/text_chunking.py")},
    "encoder_source": ENCODER_SOURCE,
    "encoder_provenance": json.loads((LOCAL_ENCODER / "source.json").read_text())
                          if (LOCAL_ENCODER / "source.json").exists() else {"model": TEXT_ENCODING["model"]},
    "text_encoding": TEXT_ENCODING, "settings": SETTINGS,
    "device": DEVICE, "python": platform.python_version(),
    "packages": {name: importlib.metadata.version(name)
                 for name in ("torch", "numpy", "pandas", "scikit-learn", "sentence-transformers", "transformers")},
    "rows": len(frame), "items": frame.item_id.nunique(),
    "evaluation": "Fixed saved sample; item/prompt-disjoint internal validation; benchmark-disjoint outer evaluation; all subjects scored.",
}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
experiments = [("item_holdout", None)] + [
    ("benchmark__" + benchmark, benchmark) for benchmark in sorted(frame.benchmark_id.unique())
]
comparisons = []
for name, benchmark in experiments:
    split = make_split(frame, SETTINGS["seed"], SETTINGS["validation_fraction"], benchmark)
    print("\n", name, {label: len(part) for label, part in split.items()})
    comparisons.append(run_fold(split, item_vectors, RUN_DIR / name, SETTINGS, name, DEVICE))
comparison = pd.concat(comparisons, ignore_index=True)
comparison.to_csv(RUN_DIR / "comparison.csv", index=False)
print("Saved all five experiments:", RUN_DIR)


## Selected-checkpoint validation scores

The baselines use constant 0.5 and the **training-only** global response mean. No validation or benchmark labels fit the baselines. Internal validation is used to choose an epoch, so these are validation scores rather than unbiased final-test estimates.


In [ ]:
# Saved results: this cell and the plots work after restarting the kernel.
comparison = pd.read_csv(RUN_DIR / "comparison.csv")
columns = ["experiment", "predictor", "rows", "best_epoch", "row_Brier", "equal_pair_Brier", "BCE",
           "known_subject_rows", "unseen_subject_rows"]
validation_scores = comparison[comparison.partition == "validation"][columns]
display(validation_scores.round(4))


## Plot 1: learning curves / overfitting

Blue is training Brier; orange is validation Brier with equal response weighting. Green gives equal weight to each subject-benchmark pair and controls checkpoint selection. The vertical line marks the restored epoch. Diverging training and validation curves suggest overfitting.


In [ ]:
fig = learning_curves(RUN_DIR)
fig.savefig(RUN_DIR / "learning_curves.png", dpi=160, bbox_inches="tight")
plt.show()


## Plot 2: validation against constant baselines


In [ ]:
fig = comparison_plot(comparison, "validation")
fig.savefig(RUN_DIR / "validation_comparison.png", dpi=160, bbox_inches="tight")
plt.show()


## Benchmark holdout results

These rows contain only the outer benchmark, scored after checkpoint selection on other benchmarks. **Unseen subject rows remain included** using the mean-ability fallback. The known-only score is supplementary and must not replace the all-response denominator. A missing known-only value means that fold has no known subjects.


In [ ]:
benchmark_scores = comparison[comparison.partition == "heldout"][columns + ["known_only_row_Brier"]]
display(benchmark_scores.round(4))
model_holdouts = benchmark_scores[benchmark_scores.predictor == "Semantic IRT"]
print("Macro benchmark Brier (row metric, each benchmark equal):", model_holdouts.row_Brier.mean())
print("Macro benchmark Brier (equal-pair metric, each benchmark equal):", model_holdouts.equal_pair_Brier.mean())


In [ ]:
# Plot 3: the outer benchmark score is never used to select the checkpoint.
fig = comparison_plot(comparison, "heldout")
fig.savefig(RUN_DIR / "benchmark_holdout_scores.png", dpi=160, bbox_inches="tight")
plt.show()


## Item-held-out validation by benchmark

This partitions the single item-holdout validation set by benchmark. It is different from holding an entire benchmark out of training.


In [ ]:
item_predictions = pd.read_csv(RUN_DIR / "item_holdout/predictions.csv")
item_validation = item_predictions[item_predictions.partition == "validation"]
rows = []
for benchmark, group in item_validation.groupby("benchmark_id"):
    for predictor, column in (("Semantic IRT", "probability"), ("constant 0.5", "constant_0.5"),
                              ("training global mean", "training_global_mean")):
        rows.append({"experiment": benchmark, "partition": "validation", "predictor": predictor,
                     "rows": len(group), **score(group, group[column])})
item_by_benchmark = pd.DataFrame(rows)
item_by_benchmark.to_csv(RUN_DIR / "item_validation_by_benchmark.csv", index=False)
display(item_by_benchmark.round(4))
fig = comparison_plot(item_by_benchmark, "validation")
fig.axes[0].set_title("Item-held-out validation, broken down by benchmark")
fig.savefig(RUN_DIR / "item_validation_by_benchmark.png", dpi=160, bbox_inches="tight")
plt.show()


## Split and subject-coverage receipts

Use the saved response identities, item/prompt groups and known/unseen flags to audit any score. The sample contains only 42 WebAgents responses and 49 ResearchCodeBench responses; those small folds can vary substantially. This is a first single-seed experiment, not evidence of reliable competition performance or an isolated chunking improvement.


In [ ]:
split_tables = []
for path in sorted(RUN_DIR.glob("*/split.csv")):
    table = pd.read_csv(path)
    table["experiment"] = path.parent.name
    split_tables.append(table)
ledger = pd.concat(split_tables, ignore_index=True)
split_counts = ledger.groupby(["experiment", "partition", "benchmark_id"]).agg(
    responses=("row_id", "size"), items=("item_id", "nunique"), item_groups=("item_group", "nunique")
)
display(split_counts)
split_counts.to_csv(RUN_DIR / "split_counts.csv")
print("Original data SHA-256:", json.loads((RUN_DIR / "manifest.json").read_text())["dataset_sha256"])
print("Results and plots:", RUN_DIR)
